# 03 - Narrative Clustering

This notebook operationalises the central claim of framing theory: the same event produces different narratives depending on who reports it. I use sentence embeddings to place every article in a high-dimensional semantic space, then cluster them to find groups of articles that discuss the same story from similar angles.

It does five things:
1. Pulls article embeddings from ChromaDB for a given topic
2. Runs KMeans at k=3, 4, 5 and evaluates each with silhouette score and inertia
3. Falls back to DBSCAN if no k produces a silhouette score above 0.3
4. Reduces embeddings to 2D with UMAP for an interactive narrative map
5. Shows per-cluster representative headlines and bias label distributions

**Topics analysed**: BJP Modi, Indian economy, Kashmir

## 1. Setup

In [1]:
import sys
import os

# Add the project root to sys.path so I can import from src/
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import umap

from src.db import get_connection
from src.chroma_store import get_collection
from src.clustering import cluster_topic

# Connect to both stores
conn = get_connection()
collection = get_collection()

print(f"SQLite articles: {conn.execute('SELECT COUNT(*) FROM articles').fetchone()[0]}")
print(f"ChromaDB embeddings: {collection.count()}")

/home/skanda_suresh/Projects/news-lens/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SQLite articles: 472
ChromaDB embeddings: 472


## 2. Available Topics

First I check which topics have enough articles to cluster meaningfully. DBSCAN needs at least a handful of points to form a cluster, and silhouette scores are meaningless on very small datasets.

In [2]:
# Show topic breakdown - skip blank topic (GDELT articles with no assigned topic)
rows = conn.execute(
    "SELECT topic, COUNT(*) as n FROM articles "
    "WHERE topic != '' GROUP BY topic ORDER BY n DESC"
).fetchall()

topics_df = pd.DataFrame(rows, columns=["topic", "article_count"])
print(topics_df.to_string(index=False))

         topic  article_count
    US economy             96
   Ukraine war             90
climate change             50


## 3. BJP Modi - Clustering

I start with "BJP Modi" because political coverage produces the clearest bias divergence - BJP-aligned and opposition-aligned outlets will frame the same events very differently.

In [ ]:
bjp = cluster_topic("BJP Modi")

print(f"Method:    {bjp['method']}")
print(f"Best k:    {bjp['best_k']}")
print(f"Silhouette: {bjp['best_silhouette']}")
print(f"Articles:  {len(bjp['headlines'])}")
print()

# Cluster size summary
for cid, info in bjp["clusters"].items():
    label = "noise" if cid == -1 else f"Cluster {cid}"
    print(f"{label}: {info['size']} articles")

## 4. Clustering Quality: Silhouette + Elbow

Even when KMeans doesn't win, I still report its metrics - they are evidence for *why* DBSCAN was chosen. Low silhouette scores across all k values mean the data doesn't form tight spherical clusters, which is typical for news articles: stories blend into each other continuously rather than forming cleanly separated groups.

**Silhouette score** measures how well each article fits its own cluster vs. the nearest other cluster. Range -1 to 1. Above 0.3 is considered meaningful structure; below 0.3 indicates overlapping or poorly separated clusters.

**Inertia** (elbow plot) measures total within-cluster spread. Lower is tighter. The elbow is the point where adding more clusters stops meaningfully reducing inertia.

In [ ]:
metrics = bjp["kmeans_metrics"]
ks = sorted(metrics.keys())
silhouettes = [metrics[k]["silhouette"] for k in ks]
inertias    = [metrics[k]["inertia"]    for k in ks]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("KMeans Evaluation - BJP Modi", fontsize=13, y=1.02)

# Silhouette scores
ax = axes[0]
ax.bar(ks, silhouettes, color="steelblue", width=0.5)
ax.axhline(y=0.3, color="red", linestyle="--", linewidth=1, label="threshold (0.3)")
ax.set_xlabel("k (number of clusters)")
ax.set_ylabel("Silhouette score")
ax.set_title("Silhouette Score by k")
ax.set_xticks(ks)
ax.legend()
ax.set_ylim(0, max(0.35, max(silhouettes) + 0.05))

for k, s in zip(ks, silhouettes):
    ax.text(k, s + 0.005, str(s), ha="center", fontsize=9)

# Elbow curve
ax = axes[1]
ax.plot(ks, inertias, marker="o", color="steelblue")
ax.set_xlabel("k (number of clusters)")
ax.set_ylabel("Inertia")
ax.set_title("Elbow Curve (Inertia by k)")
ax.set_xticks(ks)

for k, val in zip(ks, inertias):
    ax.text(k, val + 0.3, str(val), ha="center", fontsize=9)

plt.tight_layout()
plt.savefig("../data/processed/bjp_modi_kmeans_eval.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Narrative Map: UMAP Visualization

UMAP (Uniform Manifold Approximation and Projection) reduces the 384-dimensional embeddings to 2D so I can plot them. Unlike PCA, UMAP preserves local structure - articles that are similar in 384D will still be close together in 2D.

I use `metric='cosine'` to match the cosine similarity used in ChromaDB. `n_neighbors=15` balances local and global structure. `min_dist=0.1` allows a moderately compact layout.

Hover over any point to see the headline, outlet, and predicted bias label.

In [ ]:
def run_umap(embeddings: np.ndarray) -> np.ndarray:
    """Reduce embeddings to 2D for plotting."""
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        metric="cosine",
        random_state=42,
    )
    return reducer.fit_transform(embeddings)


def make_narrative_map(result: dict, topic: str) -> go.Figure:
    """
    Build an interactive Plotly scatter plot from a cluster_topic() result.
    Each point is one article. Color = cluster. Hover shows headline + outlet + bias.
    """
    coords = run_umap(result["embeddings"])

    labels    = result["labels"]
    headlines = result["headlines"]
    metadatas = result["metadatas"]

    df = pd.DataFrame({
        "x":          coords[:, 0],
        "y":          coords[:, 1],
        "cluster":    ["noise" if l == -1 else f"Cluster {l}" for l in labels],
        "headline":   [h[:80] + "..." if len(h) > 80 else h for h in headlines],
        "outlet":     [m.get("outlet", "")     for m in metadatas],
        "bias_label": [m.get("bias_label", "") for m in metadatas],
    })

    method = result["method"].upper()
    sil    = result["best_silhouette"]
    title  = f"{topic} - Narrative Map ({method}, silhouette={sil})"

    fig = px.scatter(
        df,
        x="x", y="y",
        color="cluster",
        hover_data={"headline": True, "outlet": True, "bias_label": True, "x": False, "y": False},
        title=title,
        width=850, height=550,
    )
    fig.update_traces(marker=dict(size=7, opacity=0.8))
    fig.update_layout(legend_title_text="Cluster")
    return fig


fig = make_narrative_map(bjp, "BJP Modi")
fig

## 6. Per-Cluster Analysis

The 5 most representative headlines per cluster are the articles closest to the cluster centroid - they best capture what the cluster is "about". The bias distribution shows how left/centre/right-leaning outlets are spread across each cluster.

**Important caveat**: only 26% of articles cleared the 0.85 confidence threshold. The bias labels shown here are model predictions for all articles, including lower-confidence ones. Treat the distributions as directional, not precise.

In [ ]:
for cid, info in bjp["clusters"].items():
    if cid == -1:
        print(f"Noise ({info['size']} articles - DBSCAN could not assign these to any cluster)")
        print()
        continue

    print(f"Cluster {cid} - {info['size']} articles")
    print(f"  Bias distribution: {info['bias_distribution']}")
    print(f"  Representative headlines:")
    for h in info["representative_headlines"]:
        print(f"    - {h}")
    print()

In [ ]:
def bias_distribution_chart(result: dict, topic: str) -> go.Figure:
    """
    Grouped bar chart showing bjp_aligned/opposition_aligned/neutral percentage per cluster.
    Noise cluster is excluded - bias distributions on noise are not meaningful.
    """
    bias_labels = ["bjp_aligned", "opposition_aligned", "neutral"]
    colours = {
        "bjp_aligned":        "#FF6B00",
        "opposition_aligned": "#2563EB",
        "neutral":            "#6B7280",
    }

    cluster_names = [
        f"Cluster {cid}"
        for cid in sorted(result["clusters"].keys())
        if cid != -1
    ]

    fig = go.Figure()

    for bias in bias_labels:
        values = [
            result["clusters"][cid]["bias_distribution"].get(bias, 0)
            for cid in sorted(result["clusters"].keys())
            if cid != -1
        ]
        fig.add_trace(go.Bar(
            name=bias,
            x=cluster_names,
            y=values,
            marker_color=colours[bias],
        ))

    fig.update_layout(
        barmode="group",
        title=f"{topic} - Bias Distribution per Cluster",
        xaxis_title="Cluster",
        yaxis_title="Percentage of articles (%)",
        legend_title="Bias label",
        width=700, height=400,
    )
    return fig


bias_distribution_chart(bjp, "BJP Modi")

## 7. Indian Economy - Clustering

Running the same pipeline here lets me compare whether economic coverage shows different clustering behaviour than political coverage.

In [ ]:
economy = cluster_topic("Indian economy")

print(f"Method:     {economy['method']}")
print(f"Best k:     {economy['best_k']}")
print(f"Silhouette: {economy['best_silhouette']}")
print()

for cid, info in economy["clusters"].items():
    label = "noise" if cid == -1 else f"Cluster {cid}"
    print(f"{label}: {info['size']} articles | bias: {info['bias_distribution']}")

In [ ]:
make_narrative_map(economy, "Indian Economy")

In [ ]:
print("Indian Economy - Representative Headlines per Cluster")
print()
for cid, info in economy["clusters"].items():
    if cid == -1:
        print(f"Noise: {info['size']} articles")
        print()
        continue
    print(f"Cluster {cid} ({info['size']} articles)")
    for h in info["representative_headlines"]:
        print(f"  - {h}")
    print()

In [ ]:
bias_distribution_chart(economy, "Indian Economy")

## 8. Kashmir - Clustering

Kashmir is one of the most politically charged topics in Indian media - BJP-aligned and opposition-aligned outlets are expected to frame the same events very differently. The geopolitical dimension also means international outlets will add a third perspective.

In [ ]:
kashmir = cluster_topic("Kashmir")

print(f"Method:     {kashmir['method']}")
print(f"Best k:     {kashmir['best_k']}")
print(f"Silhouette: {kashmir['best_silhouette']}")
print()

for cid, info in kashmir["clusters"].items():
    label = "noise" if cid == -1 else f"Cluster {cid}"
    print(f"{label}: {info['size']} articles | bias: {info['bias_distribution']}")

In [ ]:
make_narrative_map(kashmir, "Kashmir")

In [ ]:
print("Kashmir - Representative Headlines per Cluster")
print()
for cid, info in kashmir["clusters"].items():
    if cid == -1:
        print(f"Noise: {info['size']} articles")
        print()
        continue
    print(f"Cluster {cid} ({info['size']} articles)")
    for h in info["representative_headlines"]:
        print(f"  - {h}")
    print()

In [ ]:
bias_distribution_chart(kashmir, "Kashmir")

## 9. Cross-Topic Silhouette Comparison

A quick summary table comparing clustering quality across all three topics.

In [ ]:
summary = []
for topic_name, result in [("BJP Modi", bjp), ("Indian economy", economy), ("Kashmir", kashmir)]:
    n_articles = len(result["headlines"])
    n_clusters = len([c for c in result["clusters"] if c != -1])
    n_noise    = result["clusters"].get(-1, {}).get("size", 0)

    best_kmeans_sil = max(v["silhouette"] for v in result["kmeans_metrics"].values())

    summary.append({
        "topic":            topic_name,
        "articles":         n_articles,
        "method":           result["method"],
        "clusters_found":   n_clusters,
        "noise_articles":   n_noise,
        "best_silhouette":  result["best_silhouette"],
        "best_kmeans_sil":  best_kmeans_sil,
    })

pd.DataFrame(summary)

## 10. Observations

**On silhouette scores**: All three topics likely produced KMeans silhouette scores below 0.3, triggering the DBSCAN fallback. This is consistent with what the NLP literature shows for news article clustering: article embeddings form a continuous manifold rather than discrete spherical clusters. The same story gets covered in many different ways - there is no clean boundary between narratives.

**On bias distributions**: With Indian political topics, the clustering is most interesting when BJP-aligned and opposition-aligned outlets land in different clusters covering the same event. If this happens, the narrative map becomes a direct visual demonstration of Entman's framing theory: two clusters, same event, different realities.

**On topic contamination**: Some RSS articles may have been pulled under a topic tag without actually being about that topic. Articles that do not match the expected topic should appear as noise points in DBSCAN - the algorithm correctly identifies them as outliers rather than forcing them into a cluster.

**On the confidence caveat**: Only articles cleared the confidence threshold. The bias labels used in cluster distributions are model predictions for all articles, not just trusted ones. The distributions should be read as directional rather than precise.

**Hypothesis check**: The key hypothesis for Indian political topics is that BJP-aligned outlets (Republic World, Zee News) and opposition-aligned outlets (The Wire, Scroll) will cluster differently on the same event. DBSCAN forming multiple distinct clusters with unequal bias distributions would support this hypothesis.